# 写一个简单的编程Agent
- 实现功能：
> 在chatbot基础上加上执行循环
>
> 代码上的 差异 很少，只是增加了新的工具和一个while循环

In [11]:
import os
from openai import OpenAI
import json

# 在根目录下创建.env 文件写入API_KEY 和 BASE_URL
# 这样可以不用配置电脑环境变量（注意不要把.env提交到git上了）
# 这行会把 .env 文件里的内容加载到 os.environ 中
from dotenv import load_dotenv

load_dotenv() # 把 .env 文件里的内容加载到 os.environ 中

# 从环境变量读取，避免把 API Key 硬编码到代码里
API_KEY = os.environ.get("API_KEY")
BASE_URL = os.environ.get("BASE_URL")
if not API_KEY or not BASE_URL:
    raise RuntimeError("请先设置环境变量 API_KEY 和 BASE_URL")

### 获取用户输入
> 1. 创建一个简易的用户消息类（只包含prompt，思考模式，和思考强度）
>
> 2. 创建一个消息获取函数，每一轮对话调用该函数即可
>

In [12]:
class UserMessage:
    def __init__(self, user_msg, thinking_mode="disabled", thinking_strength = "medium"):
        self.user_msg = user_msg
        self.thinking_mode = thinking_mode
        self.thinking_strength = thinking_strength

In [14]:
# 接受用户信息
def build_user_message():

    # 1. 用户的 prompt
    user_input=input("你：")
    if user_input == "q":
        return "q"

    messages.append({"role": "user", "content": user_input})

    # 思考模式相关（是否开启思考，思考强度）
    # DS的思考模式不支持 temperature、presence_penalty、frequency_penalty 参数
    thinking_option="disabled"
    user_thinking_option = input("是否启用思考模式？(y/n): ").strip().lower()
    if user_thinking_option == "y":
        thinking_option = "enabled"

    # 2. 思考强度选择
    thinking_strength = "medium"  # 默认中等强度
    if thinking_option == "enabled":
        user_strength_option = input("选择思考强度 (low/medium/high): ").strip().lower()
        if user_strength_option in ["low", "medium", "high"]:
            thinking_strength = user_strength_option
        else:
            print("无效的强度选项，使用默认值 medium。")

    return UserMessage(user_input, thinking_option, thinking_strength)

### 工具设置
> 1. 用`json`格式描述可以使用的工具
>
> 2. 调用`tools` 包里面的工具
>
> 3. 设置工具名到函数的映射
>

In [15]:
# 用 JSON 格式描述模型可以使用的工具
from tools import regular_tools

tools = [
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "读取指定文件的内容",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "文件路径"}
                },
                "required": ["path"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "将内容写入指定文件（会覆盖已有内容）",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "文件路径"},
                    "content": {"type": "string", "description": "要写入的内容"}
                },
                "required": ["path", "content"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "run_command",
            "description": "执行一条 shell 命令并返回输出结果",
            "parameters": {
                "type": "object",
                "properties": {
                    "command": {"type": "string", "description": "要执行的命令"}
                },
                "required": ["command"]
            }
        }
    }
]

# 工具名到函数的dict映射
tool_functions = {
    "get_weather": regular_tools.get_weather,
    "read_file": regular_tools.read_file,
    "write_file": regular_tools.write_file,
    "run_command": regular_tools.run_command,}

In [16]:
def get_response(user_message, client):
    global tools
    global messages

    response = client.chat.completions.create(
        model="deepseek-flash",
        messages=messages,
        tools=tools,
        reasoning_effort=user_message.thinking_strength,
        extra_body={"thinking": {"type": user_message.thinking_mode}}
    )

    assistant_message = response.choices[0].message

    # 如果模型决定调用工具
    if assistant_message.tool_calls:
        messages.append(assistant_message)
        for tool_call in assistant_message.tool_calls:
            # arguments 是 JSON 字符串，需要解析成字典
            args = json.loads(tool_call.function.arguments)
            # **args 把字典解包成关键字参数，等价于 func(city="北京")
            func = tool_functions[tool_call.function.name]
            result = func(**args)
            print(f"  [调用工具] {tool_call.function.name}({args}) => {result}")
            # role 为 "tool" 表示这是工具返回的结果
            # tool_call_id 用来关联这条结果对应哪个工具调用
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })
        # 拿到工具结果后再调一次模型，让它生成自然语言回答
        response = client.chat.completions.create(
            model="deepseek-flash",
            messages=messages,
            reasoning_effort=user_message.thinking_strength,
            extra_body={"thinking": {"type": user_message.thinking_mode}}
        )
        assistant_message = response.choices[0].message

    messages.append({"role": "assistant", "content": assistant_message.content})
    print(f"AI: {assistant_message.content}\n")
    return assistant_message

### 基础设置
> 1. 创建 `client` 对象
> 2. 初始化聊天记录
> 3. 输入系统提示词

In [17]:
# --- Agent 主循环 ---
client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
SYSTEM_PROMPT = """你是一个编程助手。你可以读写文件和执行命令来帮用户完成编程任务。
工作流程：先理解需求，写代码，然后运行验证。如果发现错误就修复并重新运行，直到确认正确。"""

messages = [{"role": "system", "content": SYSTEM_PROMPT}]
print("编程 Agent 已启动，输入任务开始，输入 q 退出\n")

编程 Agent 已启动，输入任务开始，输入 q 退出



### 开始执行循环
> 1. 第一个while循环是实现多轮对话
> 2. 第二个while循环实现Agent Loop
> 3. 这里还是按照`chatbot.py`的逻辑来实现，有两个区别：
>   > 1）get_response函数有返回值
>   >
>   > 2）get_response可能会重复执行

In [18]:
while True:
    user_message = build_user_message()
    if user_message == "q":
        break

    while True:
        assistant_message=get_response(user_message, client)
        if not assistant_message.tool_calls:
            break

  [调用工具] write_file({'path': 'fib.py', 'content': '#!/usr/bin/env python3\n"""输出第 n 个斐波那契数。\n\n定义：F(0)=0, F(1)=1, F(n)=F(n-1)+F(n-2)\n"""\nimport sys\n\n\ndef fib(n):\n    if n < 0:\n        raise ValueError("n 必须是非负整数")\n    a, b = 0, 1\n    for _ in range(n):\n        a, b = b, a + b\n    return a\n\n\ndef main():\n    if len(sys.argv) != 2:\n        print("用法: python fib.py <n>", file=sys.stderr)\n        sys.exit(1)\n    try:\n        n = int(sys.argv[1])\n    except ValueError:\n        print("错误: n 必须是整数", file=sys.stderr)\n        sys.exit(1)\n    print(fib(n))\n\n\nif __name__ == "__main__":\n    main()\n'}) => 已写入 fib.py
  [调用工具] run_command({'command': 'cd /tmp 2>/dev/null; python3 fib.py 10'}) => 
[错误] 系统找不到指定的路径。

AI: The `cd /tmp` failed (likely Windows). Let me run it in the current directory.

<｜｜DSML｜｜ calls>
<｜｜DSML｜｜ invoke name="run_command">
<｜｜DSML｜｜ parameter name="command" string="true">python3 fib.py 10</｜｜DSML｜｜ parameter>
</｜｜DSML｜｜ invoke>
</｜｜DSML｜｜ calls>



KeyboardInterrupt: Interrupted by user